# ClipCap validation end-to-end on Google Colab

Notebook điều phối toàn bộ validation run cho năm Mapper:

1. Mount Google Drive và kiểm tra môi trường.
2. Chạy `clipcap_caption_generation.ipynb` để sinh caption.
3. Chạy `clipcap_evaluation.ipynb` để tính CIDEr, BLEU-4 và CLIPScore.
4. In bảng xếp hạng và đường dẫn artifact.

Notebook này cố định split `val`. Không dùng notebook này để tuning trên test.

## 1. Mount Drive và đặt đường dẫn

Nếu thư mục trên Drive của bạn khác mặc định, chỉ cần sửa bốn đường dẫn trong cell dưới.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/zfs-clip-image-captioning')
CHECKPOINT_ROOT = Path(
    '/content/drive/MyDrive/clipcap_colab/experiments_fixed_epoch'
)
INFERENCE_OUTPUT_BASE = Path(
    '/content/drive/MyDrive/clipcap_colab/evaluation_outputs'
)
METRICS_OUTPUT_BASE = Path(
    '/content/drive/MyDrive/clipcap_colab/metrics'
)

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'flickr8k' / 'splits' / 'val.json'
FEATURE_CACHE_PATH = (
    PROJECT_ROOT / 'data' / 'flickr8k' / 'features' / 'clip_features.pt'
)
IMAGE_DIR = PROJECT_ROOT / 'data' / 'flickr8k' / 'raw' / 'Images'

required_paths = {
    'project': PROJECT_ROOT,
    'checkpoint root': CHECKPOINT_ROOT,
    'validation manifest': MANIFEST_PATH,
}
for label, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy {label}: {path}')
if not FEATURE_CACHE_PATH.is_file() and not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        'Cần feature cache hoặc thư mục ảnh để chạy inference validation.'
    )

os.chdir(PROJECT_ROOT)
print(f'Working directory: {Path.cwd()}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')
print(f'Feature cache: {FEATURE_CACHE_PATH}')

## 2. Cài và kiểm tra dependencies

Lần đầu chạy có thể mất vài phút. Các lần sau có thể đặt `INSTALL_DEPENDENCIES = False` nếu runtime vẫn còn package. CIDEr/BLEU-4 chuẩn COCO cần Java.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        check=True,
    )
    if importlib.util.find_spec('pycocoevalcap') is None:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'pycocoevalcap'],
            check=True,
        )

if shutil.which('java') is None:
    raise RuntimeError('Java không có trong PATH; chưa thể tính CIDEr/BLEU-4')
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['java', '-version'], check=True)
print('Colab environment is ready')

## 3. Cấu hình validation experiment

Đổi `RUN_TAG` mỗi khi thay đổi cấu hình inference. Để so sánh công bằng, một cấu hình được áp dụng giống nhau cho cả năm Mapper.

In [ ]:
RUN_TAG = 'baseline_v1'
SEED = 42
SUBSETS = (
    'train_1pct',
    'train_5pct',
    'train_10pct',
    'train_25pct',
    'train_100pct',
)
MAX_NEW_TOKENS = 15
NUM_BEAMS = 5
NUM_RETURN_SEQUENCES = 5
LENGTH_PENALTY = 1.0
EARLY_STOPPING = True
IMAGE_BATCH_SIZE = 16
METRIC_BATCH_SIZE = 32

environment = {
    'ZFS_CLIP_PROJECT_ROOT': str(PROJECT_ROOT),
    'ZFS_CLIP_SPLIT_NAME': 'val',
    'ZFS_CLIP_ALLOW_TEST': '0',
    'ZFS_CLIP_RUN_TAG': RUN_TAG,
    'ZFS_CLIP_RUN_INFERENCE': '1',
    'ZFS_CLIP_SEED': str(SEED),
    'ZFS_CLIP_SUBSETS': ','.join(SUBSETS),
    'ZFS_CLIP_MANIFEST_PATH': str(MANIFEST_PATH),
    'ZFS_CLIP_CHECKPOINT_ROOT': str(CHECKPOINT_ROOT),
    'ZFS_CLIP_INFERENCE_OUTPUT_BASE': str(INFERENCE_OUTPUT_BASE),
    'ZFS_CLIP_METRICS_OUTPUT_BASE': str(METRICS_OUTPUT_BASE),
    'ZFS_CLIP_FEATURE_CACHE': str(FEATURE_CACHE_PATH),
    'ZFS_CLIP_DEVICE': 'cuda',
    'ZFS_CLIP_IMAGE_BATCH_SIZE': str(IMAGE_BATCH_SIZE),
    'ZFS_CLIP_METRIC_BATCH_SIZE': str(METRIC_BATCH_SIZE),
    'ZFS_CLIP_MAX_NEW_TOKENS': str(MAX_NEW_TOKENS),
    'ZFS_CLIP_NUM_BEAMS': str(NUM_BEAMS),
    'ZFS_CLIP_NUM_RETURN_SEQUENCES': str(NUM_RETURN_SEQUENCES),
    'ZFS_CLIP_LENGTH_PENALTY': str(LENGTH_PENALTY),
    'ZFS_CLIP_EARLY_STOPPING': '1' if EARLY_STOPPING else '0',
    'ZFS_CLIP_REFERENCES_PER_IMAGE': '5',
}
if IMAGE_DIR.is_dir():
    environment['ZFS_CLIP_IMAGE_DIR'] = str(IMAGE_DIR)
for name, value in environment.items():
    os.environ[name] = value

print(json.dumps({
    'split': 'val',
    'run_tag': RUN_TAG,
    'subsets': list(SUBSETS),
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_beams': NUM_BEAMS,
    'num_return_sequences': NUM_RETURN_SEQUENCES,
    'length_penalty': LENGTH_PENALTY,
    'early_stopping': EARLY_STOPPING,
}, indent=2))

## 4. Chạy inference rồi tính metric

Inference có resume. Nếu Colab bị ngắt, kết nối lại Drive và chạy lại notebook với đúng `RUN_TAG`; các ảnh hoàn thành sẽ được bỏ qua. Metric chỉ bắt đầu sau khi cả năm prediction file đã đầy đủ.

In [ ]:
inference_notebook = (
    PROJECT_ROOT / 'notebook' / 'clipcap' / 'clipcap_caption_generation.ipynb'
)
evaluation_notebook = (
    PROJECT_ROOT / 'notebook' / 'clipcap' / 'clipcap_evaluation.ipynb'
)
for path in (inference_notebook, evaluation_notebook):
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy notebook: {path}')

ipython = get_ipython()
print('Starting ClipCap validation inference...')
ipython.run_line_magic('run', str(inference_notebook))
print('Starting ClipCap validation metrics...')
ipython.run_line_magic('run', str(evaluation_notebook))

## 5. Đọc bảng xếp hạng cuối

CIDEr dùng để chọn cấu hình chính; BLEU-4 dùng khi CIDEr gần nhau. CLIPScore không phải tiêu chí tuning chính.

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = (
    METRICS_OUTPUT_BASE / 'val' / RUN_TAG / f'seed_{SEED}' / 'summary.json'
)
per_image_path = summary_path.parent / 'per_image_scores.csv'
if not summary_path.is_file() or not per_image_path.is_file():
    raise FileNotFoundError('Metric artifacts chưa được tạo đầy đủ')
with summary_path.open('r', encoding='utf-8') as file:
    summary = json.load(file)
ranking = pd.DataFrame(summary['results']).sort_values('rank')
display(ranking[
    ['rank', 'experiment', 'CIDEr', 'BLEU-4', 'CLIPScore']
])
print(f'Summary: {summary_path}')
print(f'Per-image scores: {per_image_path}')
print('Validation end-to-end completed')

## 6. Sau validation

Muốn thử cấu hình khác, đổi `RUN_TAG` và tham số rồi chạy lại validation. Sau khi chốt một cấu hình, dùng notebook inference riêng để chạy test với khóa test được bật, sau đó notebook metric đọc đúng test run. Không sửa tham số dựa trên metric test.